# Human Evaluation Methodologies

## নোটবুক পরিচিতি

দুটি স্বাধীন, সম্পূর্ণ স্বয়ং-সম্পূর্ণ demo, শূন্য থেকে implement করা:

1. **pairwise preference vote থেকে Elo rating।** বেশ কয়েকটি toy "model"-কে
   নির্দিষ্ট, HIDDEN true quality স্তর দেওয়া হয়। Synthetic pairwise "মানব"
   ম্যাচ ফলাফল তৈরি হয় এইভাবে: উচ্চ-গুণমানের মডেলটি true quality ফাঁক নিয়ন্ত্রিত
   probability-তে জেতে (কখনো গ্যারান্টিযুক্ত জয় নয় — ভোটগুলো সত্যিই noisy,
   হুবহু বাস্তব মানব preference data-র মতো)। Elo rating-গুলো দাবার standard
   rating সূত্র দিয়ে ম্যাচে-ম্যাচে আপডেট করা হয়, এবং আমরা দেখি noise সত্ত্বেও
   চূড়ান্ত rating-গুলো সঠিক quality RANKING পুনরুদ্ধার করে কি না।

2. **Cohen's kappa**, শূন্য থেকে হিসাব করা এবং দুটি বিপরীত synthetic annotator
   পরিস্থিতিতে প্রয়োগ করা: একটি প্রকৃত উচ্চ-চুক্তি ক্ষেত্র, এবং একটি
   imbalanced-label ক্ষেত্র যেখানে কাঁচা percent agreement উচ্চ দেখায় কিন্তু যা
   বেশিরভাগই chance/imbalance আর্টিফ্যাক্ট — দেখানো হয় যে kappa সঠিকভাবে দুটিকে
   আলাদা করে, অথচ percent agreement তা পারে না।

## কীভাবে চালাবেন

মূল file-টি হলো `example.py` — `python example.py` দিয়ে চলে। notebook-এ একই
কোড cell-by-cell চালানো হয়; শেষ cell-টি `main()` কল করে।

In [ ]:
import random

random.seed(0)

## অংশ 1: Elo rating, pairwise preference vote থেকে

Elo expected-score সূত্র, match-পরবর্তী update নিয়ম, এবং একটি noisy
"মানব preference" ভোট সিমুলেটর — নিচের তিনটি ফাংশনই এই অংশের মূল টুল।

In [ ]:
def expected_score(rating_a, rating_b):
    """standard Elo expected-score সূত্র: A-এর B-কে হারানোর probability, খাঁটিভাবে
    rating ফাঁকের একটি ফাংশন হিসেবে।"""
    return 1.0 / (1.0 + 10 ** ((rating_b - rating_a) / 400.0))


def update_elo(rating_a, rating_b, score_a, k=32):
    """score_a: A জিতলে 1.0, A হারলে 0.0, tie হলে 0.5।"""
    exp_a = expected_score(rating_a, rating_b)
    exp_b = 1.0 - exp_a
    score_b = 1.0 - score_a
    new_a = rating_a + k * (score_a - exp_a)
    new_b = rating_b + k * (score_b - exp_b)
    return new_a, new_b


def simulate_match(true_quality_a, true_quality_b):
    """একটি noisy pairwise 'মানব পছন্দ' ভোট: ভালো মডেল বেশি বার জেতে, Elo-র নিজস্ব
    expected-score ফাংশনের একই logistic রূপ দিয়ে — কিন্তু এটি গ্যারান্টিযুক্ত জয়
    নয়: true quality-তে tie হলে ফলাফল সত্যিই random হয়, এবং প্রকৃত quality ফাঁকও
    কেবল probabilistically জেতা হয়, হুবহু noisy বাস্তব মানব রায়ের মতো।"""
    p_a_wins = 1.0 / (1.0 + 10 ** ((true_quality_b - true_quality_a) / 400.0))
    return 1.0 if random.random() < p_a_wins else 0.0

## Demo 1: Elo rating noisy ভোট থেকে সত্য ranking-এ একত্রিত হওয়া

চারটি মডেল, ভিন্ন ভিন্ন লুকানো (hidden) true quality সহ। প্রতিটি মডেল একই Elo
rating (1000.0) থেকে শুরু করে — Elo-র কাছে কোনো prior তথ্য নেই। 4,000টি random
synthetic pairwise ম্যাচের পরে, Elo rating-গুলো কি true quality ranking ফিরিয়ে
আনে?

In [ ]:
def elo_demo():
    print("=" * 78)
    print("1. ELO RATING CONVERGING TO THE TRUE QUALITY RANKING FROM NOISY VOTES")
    print("=" * 78)

    # Hidden true quality স্তর (Elo update নিজে কখনো দেখে না -- এগুলো কেবল প্রতিটি
    # simulated মানব ভোটের WIN PROBABILITY নিয়ন্ত্রণ করে)।
    true_quality = {
        "Model-A (best)":    1400.0,
        "Model-B":           1250.0,
        "Model-C":           1100.0,
        "Model-D (worst)":    950.0,
    }
    models = list(true_quality.keys())

    # সবাই একই rating থেকে শুরু করে -- Elo-র কাছে মোটেও prior তথ্য নেই।
    ratings = {model: 1000.0 for model in models}

    num_matches = 4000
    print(f"{len(models)} models, hidden true quality levels: {true_quality}")
    print(f"All models start at the same Elo rating (1000.0).")
    print(f"Simulating {num_matches} random pairwise human-preference matches...\n")

    for _ in range(num_matches):
        model_a, model_b = random.sample(models, 2)
        score_a = simulate_match(true_quality[model_a], true_quality[model_b])
        ratings[model_a], ratings[model_b] = update_elo(ratings[model_a], ratings[model_b], score_a)

    ranked_by_elo = sorted(models, key=lambda m: ratings[m], reverse=True)
    ranked_by_truth = sorted(models, key=lambda m: true_quality[m], reverse=True)

    print(f"{'model':20}{'true quality':>14}{'final Elo rating':>20}")
    print("-" * 54)
    for model in ranked_by_elo:
        print(f"{model:20}{true_quality[model]:>14.0f}{ratings[model]:>20.1f}")

    correct_ranking = ranked_by_elo == ranked_by_truth
    print(f"\nRanking by true quality:  {ranked_by_truth}")
    print(f"Ranking by final Elo:     {ranked_by_elo}")
    print(f"\n-> {'MATCH' if correct_ranking else 'MISMATCH'}: after {num_matches} individually noisy pairwise")
    print("   votes (every single match was a probabilistic coin-flip, never a")
    print("   guaranteed win for the better model), the accumulated Elo ratings")
    print(f"   {'correctly recovered' if correct_ranking else 'did not fully recover'} the true quality ranking. This is exactly how a public")
    print("   leaderboard like Chatbot Arena turns millions of individually noisy")
    print("   human votes into a single stable per-model ranking: no single vote is")
    print("   trustworthy, but the accumulated 'surprise'-weighted updates are.")


elo_demo()

## অংশ 2: Cohen's kappa, শূন্য থেকে

`kappa = (p_o - p_e) / (1 - p_e)` — দুটি সম-দৈর্ঘ্যের label list থেকে হিসাব করা
হয়। তারপর দুটি synthetic পরিস্থিতি তৈরি হয়: (ক) উচ্চ-চুক্তির annotator, এবং
(খ) imbalanced-label এ উভয় annotator প্রায় সবসময়ই majority class-টি বেছে
নেয় — কোনো ভাগাভাগি করা বিচার ছাড়াই।

In [ ]:
def cohens_kappa(labels_1, labels_2):
    """kappa = (p_o - p_e) / (1 - p_e), দুটি স্বাধীন annotator-এর বরাদ্দ করা দুইটি
    সম-দৈর্ঘ্যের category label list থেকে হিসাব করা হয়।"""
    assert len(labels_1) == len(labels_2)
    n = len(labels_1)

    # p_o: observed (raw) agreement -- একই label-যুক্ত আইটেমের ভগ্নাংশ।
    p_o = sum(1 for a, b in zip(labels_1, labels_2) if a == b) / n

    # p_e: chance অনুযায়ী expected agreement, প্রতিটি annotator-এর নিজস্ব
    # marginal label distribution থেকে: p_e = sum_k P1(k) * P2(k)।
    categories = set(labels_1) | set(labels_2)
    p_e = 0.0
    for category in categories:
        p1_k = sum(1 for a in labels_1 if a == category) / n
        p2_k = sum(1 for b in labels_2 if b == category) / n
        p_e += p1_k * p2_k

    if p_e == 1.0:
        return 1.0 if p_o == 1.0 else 0.0
    kappa = (p_o - p_e) / (1.0 - p_e)
    return kappa, p_o, p_e


def make_high_agreement_labels(n=300, true_agreement_rate=0.9):
    """দুই annotator যারা একটি অন্তর্নিহিত 'true' label-এর সাথে বেশিরভাগ সময়
    একমত, balanced 3-class distribution থেকে আঁকা, এবং প্রত্যেকে স্বাধীনভাবে
    কম হারে তার সাথে (এবং তাই কখনো কখনো একে অপরের সাথে) অসম্মত হয়।"""
    categories = ["helpful", "borderline", "unhelpful"]
    true_labels = [random.choice(categories) for _ in range(n)]

    def annotate(true_label):
        if random.random() < true_agreement_rate:
            return true_label
        return random.choice([c for c in categories if c != true_label])

    annotator_1 = [annotate(t) for t in true_labels]
    annotator_2 = [annotate(t) for t in true_labels]
    return annotator_1, annotator_2


def make_imbalanced_chance_labels(n=300, safe_fraction=0.95):
    """দুই annotator যারা প্রত্যেকে প্রায় সবকিছুই 'safe' label করে -- খাঁটিভাবে
    তাদের নিজ নিজ স্বাধীন marginal distribution থেকে, কোনো ভাগাভাগি করা
    অন্তর্নিহিত রায় ছাড়াই (প্রতিটি annotator-এর label স্বাধীনভাবে আঁকা, তাই
    তাদের মধ্যে যেকোনো চুক্তি কাকতালীয়, কোনো shared assessment নয়)।"""
    def draw_label():
        return "safe" if random.random() < safe_fraction else "unsafe"

    annotator_1 = [draw_label() for _ in range(n)]
    annotator_2 = [draw_label() for _ in range(n)]
    return annotator_1, annotator_2

## Demo 2: kappa — প্রকৃত চুক্তিকে chance চুক্তি থেকে আলাদা করা

Case A: দুজন annotator 'helpfulness' (helpful/borderline/unhelpful) রেট করছেন,
প্রত্যেকে অন্তর্নিহিত true label-এর সাথে ~90% বার একমত। Case B: দুজন annotator
'safe' বনাম 'unsafe' label করছেন, প্রত্যেকে স্বাধীনভাবে ~95% বার 'safe' অনুমান
করে — লেবেলের পেছনে কোনো shared রায় নেই। kappa কি দুটি ক্ষেত্রকে সঠিকভাবে
আলাদা করবে, যেখানে কাঁচা percent agreement বিভ্রান্তিকর?

In [ ]:
def kappa_demo():
    print("\n" + "=" * 78)
    print("2. COHEN'S KAPPA: DISTINGUISHING REAL AGREEMENT FROM CHANCE AGREEMENT")
    print("=" * 78)

    print("Case A: two annotators rating 'helpfulness' (helpful/borderline/unhelpful),")
    print("each independently agreeing with an underlying true label ~90% of the time.\n")
    high_1, high_2 = make_high_agreement_labels()
    kappa_high, p_o_high, p_e_high = cohens_kappa(high_1, high_2)
    print(f"  observed agreement p_o = {p_o_high:.3f}")
    print(f"  chance agreement   p_e = {p_e_high:.3f}")
    print(f"  Cohen's kappa           = {kappa_high:.3f}")

    print("\nCase B: two annotators labeling 'safe' vs. 'unsafe', each INDEPENDENTLY")
    print("guessing 'safe' ~95% of the time with no shared judgment behind their labels.\n")
    imb_1, imb_2 = make_imbalanced_chance_labels()
    kappa_imb, p_o_imb, p_e_imb = cohens_kappa(imb_1, imb_2)
    print(f"  observed agreement p_o = {p_o_imb:.3f}")
    print(f"  chance agreement   p_e = {p_e_imb:.3f}")
    print(f"  Cohen's kappa           = {kappa_imb:.3f}")

    print(f"\n{'case':10}{'raw percent agreement':>26}{'kappa':>12}")
    print("-" * 48)
    print(f"{'A':10}{p_o_high:>26.1%}{kappa_high:>12.3f}")
    print(f"{'B':10}{p_o_imb:>26.1%}{kappa_imb:>12.3f}")

    print(f"\n-> Case B's RAW percent agreement ({p_o_imb:.1%}) looks just as strong as, or even")
    print(f"   stronger than, Case A's ({p_o_high:.1%}) -- if you only looked at percent agreement,")
    print("   the two annotators in Case B would look like reliable, consistent labelers.")
    print(f"   But Case B's kappa ({kappa_imb:.3f}) is far lower than Case A's ({kappa_high:.3f}), because")
    print(f"   Case B's high chance-agreement rate (p_e = {p_e_imb:.3f}, driven entirely by both")
    print("   annotators guessing the majority class most of the time) explains away nearly")
    print("   all of the observed agreement. Kappa correctly reveals that Case B's annotators")
    print("   share almost no real judgment -- their labels were drawn independently with no")
    print("   shared underlying assessment at all -- while Case A's high kappa reflects")
    print("   genuine, reproducible agreement above and beyond what chance would predict.")


kappa_demo()

## সবগুলো demo একসাথে: main()

`main()` দুটি demo একই ক্রমে চালায় — মূল `example.py`-তে এটি
`if __name__ == "__main__":` guard-এর ভেতরে; notebook-এ শেষ cell হিসেবে `main()`
কল করা হয়।

In [ ]:
def main():
    elo_demo()
    kappa_demo()


main()